# Track 3D dislocation-cell segmentations across strain

This notebook loads the outputs produced by `segment_111_june_all.py`, performs rigid translational registration across strain steps, finds the common overlapping mask, crops all volumes to that common region, and builds overlap-based cell correspondences.

The intended workflow is:

1. Run the batch segmentation script on all strain steps.
2. Open this notebook.
3. Check registration overlays and common mask.
4. Build overlap matrices between consecutive strain steps.
5. Classify survival / split / merge / birth / death events.


In [3]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from skimage.registration import phase_cross_correlation

# Change this if needed.
output_root = Path("~/Documents/Data/4dcells/111_june/disell_batch_output").expanduser()

dataset_order = [
    "111_cells_2_6-1pct_mosalayers_2x_redo",
    "111_cells_2_6-2pct_mosalayers_2x",
    "111_cells_2_6-3pct_mosalayers_2x",
    "111_cells_2_6-4pct_mosalayers_2x",
    "111_cells_2_6-5pct_mosalayers_2x",
    "111_cells_2_6-7pct_mosalayers_2x",
]

strain_percent = {
    "111_cells_2_6-1pct_mosalayers_2x_redo": 6.1,
    "111_cells_2_6-2pct_mosalayers_2x": 6.2,
    "111_cells_2_6-3pct_mosalayers_2x": 6.3,
    "111_cells_2_6-4pct_mosalayers_2x": 6.4,
    "111_cells_2_6-5pct_mosalayers_2x": 6.5,
    "111_cells_2_6-7pct_mosalayers_2x": 6.7,
}

print(output_root)
print(output_root.exists())

/home/adam/Documents/Data/4dcells/111_june/disell_batch_output
True


## Load segmented volumes

In [4]:
datasets = []

for name in dataset_order:
    d = output_root / name
    if not d.exists():
        print("[skip missing]", d)
        continue

    with open(d / "summary.json") as f:
        summary = json.load(f)

    item = {
        "name": name,
        "strain_percent": strain_percent.get(name, summary.get("nominal_strain_percent")),
        "path": d,
        "registered": np.load(d / "registered_volume.npy"),
        "seg_input": np.load(d / "seg_input.npy"),
        "mask": np.load(d / "mask.npy").astype(bool),
        "labels": np.load(d / "labels.npy").astype(np.int32),
        "rgb": np.load(d / "rgb_volume.npy"),
        "summary": summary,
    }
    datasets.append(item)

print("loaded:", len(datasets))
for item in datasets:
    print(item["name"], item["registered"].shape, "labels:", item["labels"].max(), "coverage:", item["summary"]["final_coverage_inside_mask"])

loaded: 6
111_cells_2_6-1pct_mosalayers_2x_redo (11, 200, 500, 2) labels: 2851 coverage: 1.0
111_cells_2_6-2pct_mosalayers_2x (11, 200, 500, 2) labels: 2809 coverage: 1.0
111_cells_2_6-3pct_mosalayers_2x (11, 200, 500, 2) labels: 2841 coverage: 1.0
111_cells_2_6-4pct_mosalayers_2x (11, 200, 500, 2) labels: 2869 coverage: 1.0
111_cells_2_6-5pct_mosalayers_2x (12, 200, 500, 2) labels: 2976 coverage: 1.0
111_cells_2_6-7pct_mosalayers_2x (11, 200, 500, 2) labels: 2806 coverage: 1.0


## Choose a registration feature

A good feature for inter-strain registration should be spatially stable. Options:

- `mask`: robust to orientation changes, but only aligns FOV boundaries;
- `channel0` or `channel1`: uses fitted orientation channels;
- `kam`: uses internal boundary-like contrast;
- `label_binary`: uses segmented region support.

Start with `channel0` or `mask`. Inspect the overlays before trusting tracking.


In [5]:
def make_registration_feature(item, mode="channel0"):
    if mode == "mask":
        return item["mask"].astype(np.float32)

    if mode == "label_binary":
        return (item["labels"] > 0).astype(np.float32)

    if mode.startswith("channel"):
        c = int(mode.replace("channel", ""))
        arr = item["registered"][..., c].astype(np.float32)
        arr = np.nan_to_num(arr, nan=np.nanmedian(arr[np.isfinite(arr)]))
        arr = (arr - np.nanmean(arr)) / (np.nanstd(arr) + 1e-8)
        arr[~item["mask"]] = 0
        return arr.astype(np.float32)

    raise ValueError(mode)

registration_mode = "channel0"

features = [make_registration_feature(item, registration_mode) for item in datasets]

for item, feat in zip(datasets, features):
    print(item["name"], feat.shape, np.nanmin(feat), np.nanmax(feat))

111_cells_2_6-1pct_mosalayers_2x_redo (11, 200, 500) -2.1551735 2.2492735
111_cells_2_6-2pct_mosalayers_2x (11, 200, 500) -2.0880969 2.205684
111_cells_2_6-3pct_mosalayers_2x (11, 200, 500) -2.3550909 2.0619135
111_cells_2_6-4pct_mosalayers_2x (11, 200, 500) -2.3753119 2.0521693
111_cells_2_6-5pct_mosalayers_2x (12, 200, 500) -2.263611 2.0092828
111_cells_2_6-7pct_mosalayers_2x (11, 200, 500) -2.2075863 2.045743


## Estimate translations to a common reference

In [6]:
ref_index = len(datasets) // 2
ref = features[ref_index]

shifts_to_ref = []
errors = []

for i, feat in enumerate(features):
    if i == ref_index:
        shift = np.zeros(3, dtype=float)
        error = 0.0
    else:
        # shift is the translation to apply to feat to match ref.
        shift, error, phasediff = phase_cross_correlation(
            ref,
            feat,
            upsample_factor=1,
            normalization=None,
        )
        shift = np.asarray(shift, dtype=float)

    shifts_to_ref.append(shift)
    errors.append(error)
    print(i, datasets[i]["name"], "shift:", shift, "error:", error)

registration_table = pd.DataFrame({
    "dataset": [d["name"] for d in datasets],
    "strain_percent": [d["strain_percent"] for d in datasets],
    "shift_z": [s[0] for s in shifts_to_ref],
    "shift_y": [s[1] for s in shifts_to_ref],
    "shift_x": [s[2] for s in shifts_to_ref],
    "registration_error": errors,
})
registration_table

0 111_cells_2_6-1pct_mosalayers_2x_redo shift: [  5.   5. -23.] error: 0.9913447
1 111_cells_2_6-2pct_mosalayers_2x shift: [ -5.   6. -17.] error: 0.9803023
2 111_cells_2_6-3pct_mosalayers_2x shift: [-4.  3. -2.] error: 0.8803492
3 111_cells_2_6-4pct_mosalayers_2x shift: [0. 0. 0.] error: 0.0


ValueError: images must be same shape

## Apply translations and compute common overlap mask

In [ ]:
def shift_volume(arr, shift, order, cval):
    """Shift a 3D or 4D array. For 4D, shift spatial axes only."""
    if arr.ndim == 3:
        return ndi.shift(arr, shift=shift, order=order, mode="constant", cval=cval, prefilter=False)
    if arr.ndim == 4:
        out = np.empty_like(arr)
        for c in range(arr.shape[-1]):
            out[..., c] = ndi.shift(arr[..., c], shift=shift, order=order, mode="constant", cval=cval, prefilter=False)
        return out
    raise ValueError(arr.shape)

aligned = []

for item, shift in zip(datasets, shifts_to_ref):
    a = {
        "name": item["name"],
        "strain_percent": item["strain_percent"],
        "registered": shift_volume(item["registered"], shift, order=1, cval=np.nan),
        "seg_input": shift_volume(item["seg_input"], shift, order=1, cval=0.0),
        "mask": shift_volume(item["mask"].astype(np.uint8), shift, order=0, cval=0).astype(bool),
        "labels": shift_volume(item["labels"].astype(np.int32), shift, order=0, cval=0).astype(np.int32),
        "rgb": shift_volume(item["rgb"], shift, order=1, cval=0.0),
    }
    aligned.append(a)

common_mask = np.logical_and.reduce([a["mask"] for a in aligned])

print("common mask voxels:", np.count_nonzero(common_mask))
print("common fraction vs each:")
for a in aligned:
    print(a["name"], np.count_nonzero(common_mask) / max(np.count_nonzero(a["mask"]), 1))

In [ ]:
def bounding_box(mask, pad=0):
    coords = np.argwhere(mask)
    if coords.size == 0:
        raise ValueError("empty mask")
    lo = np.maximum(coords.min(axis=0) - pad, 0)
    hi = np.minimum(coords.max(axis=0) + pad + 1, mask.shape)
    return tuple(slice(int(l), int(h)) for l, h in zip(lo, hi))

crop = bounding_box(common_mask, pad=0)
print(crop)

for a in aligned:
    for key in ["registered", "seg_input", "mask", "labels", "rgb"]:
        a[key + "_crop"] = a[key][crop] if a[key].ndim == 3 else a[key][crop + (slice(None),)]

common_mask_crop = common_mask[crop]
print("cropped common mask:", common_mask_crop.shape, np.count_nonzero(common_mask_crop))

## Visual check of aligned segmentations

In [ ]:
z = common_mask_crop.shape[0] // 2

fig, axes = plt.subplots(len(aligned), 3, figsize=(14, 3 * len(aligned)))

if len(aligned) == 1:
    axes = axes[None, :]

for row, a in zip(axes, aligned):
    row[0].imshow(a["rgb_crop"][z], aspect=2.8)
    row[0].set_title(f"{a['strain_percent']}% RGB")
    row[0].axis("off")

    row[1].imshow(a["labels_crop"][z], cmap="nipy_spectral", interpolation="nearest", aspect=2.8)
    row[1].set_title("labels")
    row[1].axis("off")

    row[2].imshow(common_mask_crop[z], cmap="gray", interpolation="nearest", aspect=2.8)
    row[2].set_title("common mask")
    row[2].axis("off")

plt.tight_layout()
plt.show()

## Overlap-based cell tracking

The tracking below is deliberately conservative. It does not assume perfect identity across strain. It builds an overlap matrix between consecutive label volumes after registration/cropping.

For a label \(i\) at strain \(t\), candidate descendants at \(t+\Delta t\) are labels \(j\) with non-zero spatial overlap. We classify:

- `one_to_one`: one dominant parent and one dominant child;
- `split`: one parent overlaps significantly with multiple children;
- `merge`: multiple parents overlap significantly with one child;
- `death`: parent has no sufficient child overlap;
- `birth`: child has no sufficient parent overlap.

You must inspect examples. Registration/segmentation inconsistency can mimic topological events.


In [ ]:
def overlap_table(labels_a, labels_b, mask, min_overlap_voxels=10):
    a = labels_a[mask].astype(np.int64)
    b = labels_b[mask].astype(np.int64)

    keep = (a > 0) & (b > 0)
    a = a[keep]
    b = b[keep]

    if len(a) == 0:
        return pd.DataFrame(columns=["label_a", "label_b", "overlap_voxels"])

    pairs = np.stack([a, b], axis=1)
    unique_pairs, counts = np.unique(pairs, axis=0, return_counts=True)

    df = pd.DataFrame({
        "label_a": unique_pairs[:, 0].astype(int),
        "label_b": unique_pairs[:, 1].astype(int),
        "overlap_voxels": counts.astype(int),
    })

    df = df[df["overlap_voxels"] >= min_overlap_voxels].copy()
    return df.sort_values("overlap_voxels", ascending=False).reset_index(drop=True)


def classify_pair_tracking(labels_a, labels_b, mask, min_overlap_voxels=10, min_fraction=0.10):
    overlaps = overlap_table(labels_a, labels_b, mask, min_overlap_voxels=min_overlap_voxels)

    size_a = pd.Series(labels_a[mask][labels_a[mask] > 0]).value_counts().rename_axis("label_a").rename("size_a")
    size_b = pd.Series(labels_b[mask][labels_b[mask] > 0]).value_counts().rename_axis("label_b").rename("size_b")

    overlaps = overlaps.merge(size_a, on="label_a", how="left")
    overlaps = overlaps.merge(size_b, on="label_b", how="left")
    overlaps["frac_of_a"] = overlaps["overlap_voxels"] / overlaps["size_a"]
    overlaps["frac_of_b"] = overlaps["overlap_voxels"] / overlaps["size_b"]

    significant = overlaps[
        (overlaps["frac_of_a"] >= min_fraction) |
        (overlaps["frac_of_b"] >= min_fraction)
    ].copy()

    children_per_parent = significant.groupby("label_a")["label_b"].nunique()
    parents_per_child = significant.groupby("label_b")["label_a"].nunique()

    events = []

    all_a = set(size_a.index.astype(int))
    all_b = set(size_b.index.astype(int))

    tracked_a = set(significant["label_a"].astype(int))
    tracked_b = set(significant["label_b"].astype(int))

    for lab_a in sorted(all_a):
        n_child = int(children_per_parent.get(lab_a, 0))
        if n_child == 0:
            events.append({"event": "death", "label_a": lab_a, "label_b": 0, "n_children": 0, "n_parents": np.nan})
        elif n_child == 1:
            child = int(significant.loc[significant["label_a"] == lab_a, "label_b"].iloc[0])
            n_parent = int(parents_per_child.get(child, 0))
            event = "one_to_one" if n_parent == 1 else "merge_candidate"
            events.append({"event": event, "label_a": lab_a, "label_b": child, "n_children": n_child, "n_parents": n_parent})
        else:
            events.append({"event": "split", "label_a": lab_a, "label_b": 0, "n_children": n_child, "n_parents": np.nan})

    for lab_b in sorted(all_b - tracked_b):
        events.append({"event": "birth", "label_a": 0, "label_b": lab_b, "n_children": np.nan, "n_parents": 0})

    events = pd.DataFrame(events)
    return overlaps, significant, events

In [ ]:
tracking_outputs = []

tracking_dir = output_root / "tracking_output"
tracking_dir.mkdir(exist_ok=True)

for i in range(len(aligned) - 1):
    a = aligned[i]
    b = aligned[i + 1]

    overlaps, significant, events = classify_pair_tracking(
        a["labels_crop"],
        b["labels_crop"],
        common_mask_crop,
        min_overlap_voxels=10,
        min_fraction=0.10,
    )

    pair_name = f"{i:02d}_{a['strain_percent']}_to_{b['strain_percent']}"
    overlaps.to_csv(tracking_dir / f"{pair_name}_overlaps.csv", index=False)
    significant.to_csv(tracking_dir / f"{pair_name}_significant_overlaps.csv", index=False)
    events.to_csv(tracking_dir / f"{pair_name}_events.csv", index=False)

    summary = events["event"].value_counts().rename_axis("event").rename("count").reset_index()
    summary["from"] = a["strain_percent"]
    summary["to"] = b["strain_percent"]
    tracking_outputs.append(summary)

    print("\n", pair_name)
    print(summary)

tracking_summary = pd.concat(tracking_outputs, ignore_index=True)
tracking_summary.to_csv(tracking_dir / "tracking_event_summary.csv", index=False)
tracking_summary

## Event fractions versus strain

In [ ]:
pivot = tracking_summary.pivot_table(
    index=["from", "to"],
    columns="event",
    values="count",
    fill_value=0,
)

pivot_fraction = pivot.div(pivot.sum(axis=1), axis=0)

ax = pivot_fraction.plot(kind="bar", stacked=True, figsize=(10, 5))
ax.set_ylabel("event fraction")
ax.set_title("Overlap-tracking event fractions")
plt.tight_layout()
plt.show()

pivot, pivot_fraction

## Inspect candidate split events

In [ ]:
# Pick a transition and inspect the largest split candidates.
transition_index = 0

a = aligned[transition_index]
b = aligned[transition_index + 1]

events = pd.read_csv(tracking_dir / f"{transition_index:02d}_{a['strain_percent']}_to_{b['strain_percent']}_events.csv")
overlaps = pd.read_csv(tracking_dir / f"{transition_index:02d}_{a['strain_percent']}_to_{b['strain_percent']}_significant_overlaps.csv")

split_labels = events.loc[events["event"] == "split", "label_a"].astype(int).to_numpy()
print("number of split candidates:", len(split_labels))

if len(split_labels):
    lab = split_labels[0]
    children = overlaps.loc[overlaps["label_a"] == lab, "label_b"].astype(int).to_numpy()
    print("parent:", lab, "children:", children)

    parent_mask = a["labels_crop"] == lab
    child_mask = np.isin(b["labels_crop"], children)

    coords = np.argwhere(parent_mask | child_mask)
    z = int(np.median(coords[:, 0])) if len(coords) else common_mask_crop.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(a["labels_crop"][z] == lab, cmap="gray", interpolation="nearest", aspect=2.8)
    axes[0].set_title(f"parent {lab}, z={z}")
    axes[0].axis("off")

    axes[1].imshow(np.isin(b["labels_crop"][z], children), cmap="gray", interpolation="nearest", aspect=2.8)
    axes[1].set_title("children")
    axes[1].axis("off")

    overlay = np.zeros((*parent_mask[z].shape, 3), dtype=float)
    overlay[..., 0] = parent_mask[z]
    overlay[..., 1] = np.isin(b["labels_crop"][z], children)
    axes[2].imshow(overlay, interpolation="nearest", aspect=2.8)
    axes[2].set_title("red=parent, green=children")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

## Save aligned cropped arrays for downstream analysis

In [ ]:
aligned_dir = output_root / "aligned_common_crop"
aligned_dir.mkdir(exist_ok=True)

np.save(aligned_dir / "common_mask_crop.npy", common_mask_crop)
registration_table.to_csv(aligned_dir / "registration_table.csv", index=False)

for a in aligned:
    safe = str(a["strain_percent"]).replace(".", "p")
    np.save(aligned_dir / f"{safe}_labels_crop.npy", a["labels_crop"])
    np.save(aligned_dir / f"{safe}_registered_crop.npy", a["registered_crop"])
    np.save(aligned_dir / f"{safe}_mask_crop.npy", a["mask_crop"])
    np.save(aligned_dir / f"{safe}_rgb_crop.npy", a["rgb_crop"])

print("saved:", aligned_dir)